In [ ]:
%pip install eyepop==3.12.0

In [ ]:
import getpass

EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

In [ ]:
NAMESPACE_PREFIX="XXXXXXXXXXX" # Add your namespace-prefix here

### Define Ability

In [28]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import CropForward, ForwardComponent, FullForward, InferenceComponent, Pop
import json


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.describe.product-placement",
        description="Identify brand product placement in the images",
        worker_release="qwen3-instruct",
        text_prompt="""
            You are given an image of a social media post. Your task is to visually inspect the image and determine if a specific target product is present.

            TARGET PRODUCT: A bright neon-blue aluminum soda can featuring a bold yellow lightning-bolt logo on the front.

            Return ONLY valid JSON.
            Do not include explanation.
            Do not include markdown formatting (do not use ```json).
            Do not include commentary.

            ---------------------------------------
            INSTRUCTIONS:

            1. Visually scan the entire image for the TARGET PRODUCT based on its physical appearance (neon-blue can, yellow lightning bolt).
            2. If the TARGET PRODUCT is visible in the image, set "target_product" to "found".
            3. If the TARGET PRODUCT is NOT visible in the image, set "target_product" to "not_found".
            4. In the "description" field, describe exactly where the TARGET PRODUCT is located in the image and how it is being used. If the product is not found, set "description" to null.
            5. Do NOT guess or infer the product's presence if it is hidden or obscured.
            6. Do NOT fabricate data.

            ---------------------------------------
            RETURN THIS EXACT JSON STRUCTURE:

            {
              "target_product": "found",
              "description": "String describing the product's location and use, or null."
            }
            ---------------------------------------

        """,
        transform_into=TransformInto(),
       config=InferRuntimeConfig(
           max_new_tokens=450,
           image_size=512
       ),
       is_public=False
   )
]




### Create Ability

In [ ]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

### Evalulate on a Single Image

In [ ]:
from pathlib import Path

pop = Pop(components=[
   InferenceComponent(
       ability=f"{NAMESPACE_PREFIX}.describe.product-placement:latest",
   )
])

img_path = "content/sample_img.png" # Add path to image

with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
   endpoint.set_pop(pop)
   sample_img_path = Path(img_path)
   job = endpoint.upload(sample_img_path)
   while result := job.predict():
      print(json.dumps(result, indent=2))

print("Done")